In [0]:
# Import packages
import requests
import io
import pandas as pd
import seaborn as sbn
import matplotlib.pyplot as plt

from datetime import date, timedelta
from IPython.display import display, Markdown

In [0]:
###################
#SET E-MAIL HEADER#
###################

# This cell will serve as a header in your email. You can add some information here that may be useful for providing context. 
# If you want to change the actual text that appears, feel free to edit the "md_text" variable directly.

# Input a title of your choosing here.
title = "Weekly Digest | 311 in Downtown Cleveland"

# Write a brief description about the analysis.
description = """
This email includes the following:
* The number of 311 requests within the last 24 hours, 7 days, and 30 days.
* Trend of service request counts over the last 30 days.
* The top 10 most common requests within the 30 days.
* Requests from last week that have been closed this week.
* Longest "still open" requests.
"""

# Get today's date
current_date = date.today()

# Print markdown header
md_text = f"""
## {title}
**Description**
{description}
**Data as of**  
{current_date}
"""

# Render it in the output
display(Markdown(md_text))

In [0]:
###########
#PULL DATA#
###########

# Put the URL for your API request here. You can do this using the query builder in ArcGIS Online.
url = "https://services3.arcgis.com/dty2kHktVXHrqO8i/arcgis/rest/services/Data_311/FeatureServer/0/query"

# Type out a where clause here. 
# You can utilize an "f string" to make this filter dynamic.
# NOTE: For many feature layers, the maximum amount of records the ArcGIS Online API can query is 2,000. You'll need to perform multiple queries if you are reading in more than 2k records.
days_ago_30 = str(current_date - timedelta(days=30))
where_clause = f"""
requested_datetime >= '{days_ago_30}' AND neighborhood = 'Downtown'
"""

# Set query parameters. Nothing here for you to do.
query_params = {
    "where": where_clause,       # The where clause from above.
    "returnGeometry": "false",    # We're not doing any work with spatial data. But if you want to make maps with your data, set to 'true'.
    "f": "json"                  # Tells the server to respond with JSON format.
}

## SUBMIT AND PARSE API REQUEST(S)
# Since in many cases we are limited to reading 2,000 records per query, this function will make several API requests to get all the records.
# It will also parse the request and extract the data into a list of records.
def get_data(url,query_params):
    """
    This function retrieves data from ArcGIS Online FeatureLayer via a series of GET requests. 
    It will pull data in groups of 2,000 records, and append all the data to one list of records.

    Args:
        url (str): Spark session context.
        query_params (dict): A spark DataFrame to geocode.

    Returns:
        list: The spark DataFrame, with new geocoded columns appended.
    """
    # Get record count
    record_count = requests.get(url,params={"where":query_params['where'],"returnCountOnly":"true","f":"json"}).json()['count']

    # Split record count into offsets
    offsets = range(0,record_count,2000)
    
    # List of data records
    results = []
    
    # Loop through offsets and get data for each offset
    for i,offset in enumerate(offsets):
        # Perform a GET request with the given offset
        query_params['resultOffset'] = offset
        req = requests.get(url,params=query_params)
        # Get JSON
        resp = req.json()

        # Extract data and append it to our final result
        data = [a['attributes'] for a in resp['features']]

        results += data
    
    # Ensure the number of records matches the record count of the Feature Layer.
    assert len(results) == record_count

    return results

data = get_data(url, query_params)

In [0]:
######################
#CONVERT TO DATAFRAME#
######################
# If you use the get_data function from above, your data should look something like this:.
"""
[{'service_request_id': '202000403109',
  'service_category': 'Trash & Recycling',
  'service_name': 'Waste Cart Concerns'},
 {'service_request_id': '202000403083',
  'service_category': 'Building & Housing',
  'service_name': 'Electrical Issue'},
 {'service_request_id': '202000403082',
  'service_category': 'Street Issues',
  'service_name': 'Debris in Street'}]
"""

# The format above is known as "records" format, and will allow you to automatically convert to a pandas dataframe when doing pd.DataFrame(records).
# Try converting to DataFrame below:
df = pd.DataFrame(data)

# Select the columns that we'll use for analysis
columns_to_select = [
  'service_request_id',
  'service_category',
  'service_name',
  'status_description',
  'requested_datetime',
  'closed_date'
]
df = df[columns_to_select]

# The ArcGIS API returns date columns as integers, we should convert these to dates.
df['requested_datetime'] = pd.to_datetime(df['requested_datetime'],unit='ms')
df['closed_date'] = pd.to_datetime(df['closed_date'],unit='ms')

In [0]:
##########
#ANALYSIS#
##########

# Conduct your data analysis below. You should use the dataframe from above as your starting point. Feel free to add more cells to separate output.
# We import the "seaborn" package in the first cell above. You can use this or another package of your choosing for creating visualizations.
# If you choose to import additional packages, be sure to update the dependencies in your GitHub Action! Otherwise the workflow will fail.
days_ago_7 = current_date - timedelta(days=7)
days_ago_1 = current_date - timedelta(days=1)

num_last_30_days = df.shape[0]

mask = df['requested_datetime'].dt.date >= days_ago_7
num_last_7_days = df[mask].shape[0]

mask = df['requested_datetime'].dt.date >= days_ago_1
num_last_day = df[mask].shape[0]

md_text = f"""
## Service Request Totals
**New Service Requests**  
Last 30 Days: **{num_last_30_days}**  
Last 7 Days: **{num_last_7_days}**  
Last 1 Day: **{num_last_day}**
"""

display(Markdown(md_text))

In [0]:
# Now let's create a trendline showing how the volume of calls have changed over time.
# First we need to get our data grouped by counts of calls by day
counts_by_month = df.groupby(df['requested_datetime'].dt.date)['service_request_id']\
    .count()\
    .sort_index()

# Create the lineplot
sbn.lineplot(counts_by_month)

# Specify design elements of the lineplot
plt.xticks(rotation=20)
plt.xlabel('Date')
plt.ylabel('Number of Calls')

# Prep output
md_text = f"""
## Call Trend, {days_ago_30} to {str(current_date)} 
"""

display(Markdown(md_text))
plt.show()

In [0]:
# Get the number of service requests by category.
counts_by_category = df.groupby([df['service_category']])['service_request_id']\
    .count()\
    .sort_values(ascending=False)\
    .reset_index()

# Plot it on a bar plot!
sbn.barplot(counts_by_category,x='service_category',y='service_request_id',hue='service_category')
plt.xticks(rotation=65,fontsize=10)
plt.xlabel('Service Category')
plt.ylabel('Number of Service Requests')

# Get the number of service requests by name of service. We'll display this as a table below.
counts_by_service = df.groupby([df['service_name']])['service_request_id']\
    .count()\
    .sort_values(ascending=False)\
    .reset_index()\
    .rename(columns={"service_name":"Service Name","service_request_id":"Number of Service Requests"})

# Prep output
md_text = f"""
## Service Requests by Category 
"""
display(Markdown(md_text))
plt.show()
md_text = f"""
## Service Request Overview 
"""
display(Markdown(md_text))
counts_by_service